In [6]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error

In [7]:
DATA_PATH = Path(
    "../data/processed/hdb_resale_clean.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["month"]
)

print(
    "Rows:",
    len(df)
)

print(
    "Date range:",
    df["month"].min(),
    "to",
    df["month"].max()
)

Rows: 244230
Date range: 2017-01-01 00:00:00 to 2026-11-01 00:00:00


In [8]:
MODEL_PATH = Path(
    "../models/final/hdb_resale_xgboost_calibrated.joblib"
)

model = joblib.load(
    MODEL_PATH
)

print(
    "Loaded model:",
    MODEL_PATH
)

Loaded model: ..\models\final\hdb_resale_xgboost_calibrated.joblib


In [9]:
CALIBRATION_PATH = Path(
    "../models/final/hdb_resale_calibration.json"
)

with open(
    CALIBRATION_PATH,
    "r"
) as file:

    calibration_metadata = json.load(
        file
    )

EVIDENCE_RULE = (
    calibration_metadata[
        "evidence_rule"
    ]
)

INTERVAL_MARGIN = (
    calibration_metadata[
        "interval"
    ][
        "interval_margin"
    ]
)

print(
    "Evidence rule:",
    EVIDENCE_RULE
)

print(
    "Interval margin:",
    INTERVAL_MARGIN
)

Evidence rule: {'min_candidate_count': 10, 'min_top5_avg_score': 75, 'min_top5_strong_count': 2, 'strong_score_threshold': 75}
Interval margin: 65339.25


In [10]:
final_df = df[
    df["month"]
    == "2026-10-01"
].copy()

print(
    "Final evaluation rows:",
    len(final_df)
)

print(
    "Final evaluation month:",
    final_df["month"].min()
)

Final evaluation rows: 2175
Final evaluation month: 2026-10-01 00:00:00


In [11]:
FEATURES = [
    "town",
    "flat_type",
    "storey_mid",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]

TARGET = "resale_price"

X_final = final_df[
    FEATURES
].copy()

y_final = final_df[
    TARGET
].copy()

print(
    "X_final:",
    X_final.shape
)

X_final: (2175, 8)


In [12]:
final_predictions = model.predict(
    X_final
)

final_results = final_df.copy()

final_results[
    "predicted_price"
] = final_predictions

final_results[
    "absolute_error"
] = np.abs(
    final_results[
        "resale_price"
    ]
    - final_results[
        "predicted_price"
    ]
)

final_results[
    "ape"
] = (
    final_results[
        "absolute_error"
    ]
    / final_results[
        "resale_price"
    ]
    * 100
)

In [13]:
overall_mae = mean_absolute_error(
    y_final,
    final_predictions
)

overall_median_ape = (
    final_results[
        "ape"
    ].median()
)

print(
    "Overall MAE:",
    round(
        overall_mae,
        2
    )
)

print(
    "Overall Median APE:",
    round(
        overall_median_ape,
        4
    ),
    "%"
)

Overall MAE: 34564.78
Overall Median APE: 3.9082 %


In [14]:
def get_comparable_candidates(
    target_row,
    history_df
):
    """
    Return same-town, same-flat-type transactions
    from the previous 12 months.
    """

    target_month = (
        target_row["month"]
    )

    start_month = (
        target_month
        - pd.DateOffset(
            months=12
        )
    )

    candidates = history_df[
        (
            history_df["town"]
            == target_row["town"]
        )
        &
        (
            history_df["flat_type"]
            == target_row["flat_type"]
        )
        &
        (
            history_df["month"]
            < target_month
        )
        &
        (
            history_df["month"]
            >= start_month
        )
    ].copy()

    return candidates

In [15]:
def score_comparables(
    target_row,
    candidates
):
    scored = candidates.copy()

    scored["same_block"] = (
        scored["block"]
        == target_row["block"]
    ).astype(int)

    scored["same_street"] = (
        scored["street_name"]
        == target_row["street_name"]
    ).astype(int)

    scored["same_model"] = (
        scored["flat_model"]
        == target_row["flat_model"]
    ).astype(int)

    scored["area_diff"] = np.abs(
        scored["floor_area_sqm"]
        - target_row["floor_area_sqm"]
    )

    scored["lease_diff"] = np.abs(
        scored["remaining_lease_months"]
        - target_row[
            "remaining_lease_months"
        ]
    )

    scored["storey_diff"] = np.abs(
        scored["storey_mid"]
        - target_row["storey_mid"]
    )

    scored["months_ago"] = (
        (
            target_row["month"].year
            - scored["month"].dt.year
        ) * 12
        +
        (
            target_row["month"].month
            - scored["month"].dt.month
        )
    )

    scored["area_score"] = (
        1
        - scored["area_diff"] / 20
    ).clip(
        lower=0,
        upper=1
    )

    scored["lease_score"] = (
        1
        - scored["lease_diff"] / 120
    ).clip(
        lower=0,
        upper=1
    )

    scored["storey_score"] = (
        1
        - scored["storey_diff"] / 15
    ).clip(
        lower=0,
        upper=1
    )

    scored["recency_score"] = (
        1
        - scored["months_ago"] / 12
    ).clip(
        lower=0,
        upper=1
    )

    scored[
        "comparable_score"
    ] = (
        scored["same_block"] * 20
        + scored["same_street"] * 15
        + scored["same_model"] * 10
        + scored["area_score"] * 20
        + scored["lease_score"] * 15
        + scored["storey_score"] * 10
        + scored["recency_score"] * 10
    )

    return (
        scored
        .sort_values(
            "comparable_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

In [16]:
def calculate_evidence_summary(
    target_row,
    history_df
):
    candidates = (
        get_comparable_candidates(
            target_row,
            history_df
        )
    )

    if candidates.empty:

        return {
            "candidate_count": 0,
            "top5_avg_score": np.nan,
            "top5_strong_count": 0,
            "top1_score": np.nan
        }

    scored = score_comparables(
        target_row,
        candidates
    )

    top5 = scored.head(5)

    strong_threshold = (
        EVIDENCE_RULE[
            "strong_score_threshold"
        ]
    )

    return {
        "candidate_count":
            len(scored),

        "top5_avg_score":
            top5[
                "comparable_score"
            ].mean(),

        "top5_strong_count":
            (
                top5[
                    "comparable_score"
                ]
                >= strong_threshold
            ).sum(),

        "top1_score":
            scored[
                "comparable_score"
            ].iloc[0]
    }

In [17]:
history_df = df[
    df["month"]
    < "2026-10-01"
].copy()

print(
    "History rows:",
    len(history_df)
)

print(
    "History through:",
    history_df[
        "month"
    ].max()
)

History rows: 241697
History through: 2026-09-01 00:00:00


In [18]:
evidence_summaries = []

for index, target_row in (
    final_df.iterrows()
):

    summary = (
        calculate_evidence_summary(
            target_row,
            history_df
        )
    )

    summary["index"] = index

    evidence_summaries.append(
        summary
    )


evidence_df = (
    pd.DataFrame(
        evidence_summaries
    )
    .set_index(
        "index"
    )
)

print(
    "Evidence rows:",
    len(evidence_df)
)

Evidence rows: 2175


In [19]:
final_results = (
    final_results
    .join(
        evidence_df
    )
)

final_results[
    [
        "resale_price",
        "predicted_price",
        "absolute_error",
        "candidate_count",
        "top5_avg_score",
        "top5_strong_count",
        "top1_score"
    ]
].head()

,resale_price,predicted_price,absolute_error,candidate_count,top5_avg_score,top5_strong_count,top1_score
222076,290000.0,306096.40625,16096.40625,23,92.016667,5,97.250000
222077,358000.0,328990.53125,29009.46875,23,86.675000,4,93.958333
222084,323000.0,335002.59375,12002.59375,23,83.200000,5,97.250000
222085,285000.0,300858.93750,15858.93750,23,69.125000,1,95.041667
222128,412000.0,421922.81250,9922.81250,421,87.550000,5,96.083333


In [20]:
final_results[
    "evidence_sufficient"
] = (
    (
        final_results[
            "candidate_count"
        ]
        >= EVIDENCE_RULE[
            "min_candidate_count"
        ]
    )
    &
    (
        final_results[
            "top5_avg_score"
        ]
        >= EVIDENCE_RULE[
            "min_top5_avg_score"
        ]
    )
    &
    (
        final_results[
            "top5_strong_count"
        ]
        >= EVIDENCE_RULE[
            "min_top5_strong_count"
        ]
    )
)

In [21]:
accepted_count = (
    final_results[
        "evidence_sufficient"
    ].sum()
)

total_count = len(
    final_results
)

abstained_count = (
    total_count
    - accepted_count
)

accepted_pct = (
    accepted_count
    / total_count
    * 100
)

abstention_pct = (
    abstained_count
    / total_count
    * 100
)

print(
    "Accepted:",
    accepted_count
)

print(
    "Abstained:",
    abstained_count
)

print(
    "Accepted %:",
    round(
        accepted_pct,
        2
    )
)

print(
    "Abstention %:",
    round(
        abstention_pct,
        2
    )
)

Accepted: 1500
Abstained: 675
Accepted %: 68.97
Abstention %: 31.03


In [22]:
accepted_results = (
    final_results[
        final_results[
            "evidence_sufficient"
        ]
    ].copy()
)

accepted_mae = (
    accepted_results[
        "absolute_error"
    ].mean()
)

accepted_median_ape = (
    accepted_results[
        "ape"
    ].median()
)

print(
    "Accepted-case MAE:",
    round(
        accepted_mae,
        2
    )
)

print(
    "Accepted-case Median APE:",
    round(
        accepted_median_ape,
        4
    ),
    "%"
)

Accepted-case MAE: 32761.14
Accepted-case Median APE: 3.8558 %


In [23]:
accepted_results[
    "interval_lower"
] = (
    accepted_results[
        "predicted_price"
    ]
    - INTERVAL_MARGIN
)

accepted_results[
    "interval_upper"
] = (
    accepted_results[
        "predicted_price"
    ]
    + INTERVAL_MARGIN
)

In [24]:
accepted_results[
    "interval_covered"
] = (
    (
        accepted_results[
            "resale_price"
        ]
        >= accepted_results[
            "interval_lower"
        ]
    )
    &
    (
        accepted_results[
            "resale_price"
        ]
        <= accepted_results[
            "interval_upper"
        ]
    )
)

final_interval_coverage = (
    accepted_results[
        "interval_covered"
    ].mean()
    * 100
)

interval_width = (
    INTERVAL_MARGIN
    * 2
)

print(
    "Final interval coverage:",
    round(
        final_interval_coverage,
        2
    ),
    "%"
)

print(
    "Interval margin:",
    round(
        INTERVAL_MARGIN,
        2
    )
)

print(
    "Interval width:",
    round(
        interval_width,
        2
    )
)

Final interval coverage: 88.13 %
Interval margin: 65339.25
Interval width: 130678.5


In [25]:
flat_type_results = (
    accepted_results
    .groupby(
        "flat_type"
    )
    .agg(
        cases=(
            "resale_price",
            "size"
        ),
        mae=(
            "absolute_error",
            "mean"
        ),
        median_ape_pct=(
            "ape",
            "median"
        ),
        interval_coverage=(
            "interval_covered",
            "mean"
        )
    )
    .reset_index()
)

flat_type_results[
    "interval_coverage"
] *= 100

flat_type_results

,flat_type,cases,mae,median_ape_pct,interval_coverage
0,2 ROOM,37,16187.706081,3.679348,97.297297
1,3 ROOM,360,21124.186632,3.539598,97.777778
2,4 ROOM,635,33138.464124,3.859693,87.874016
3,5 ROOM,376,40702.925519,4.060486,81.648936
4,EXECUTIVE,92,49900.394701,4.157157,75.000000


In [26]:
town_results = (
    accepted_results
    .groupby(
        "town"
    )
    .agg(
        cases=(
            "resale_price",
            "size"
        ),
        mae=(
            "absolute_error",
            "mean"
        ),
        median_ape_pct=(
            "ape",
            "median"
        ),
        interval_coverage=(
            "interval_covered",
            "mean"
        )
    )
    .reset_index()
)

town_results[
    "interval_coverage"
] *= 100

town_results.sort_values(
    "mae"
)

,town,cases,mae,median_ape_pct,interval_coverage
3,BUKIT BATOK,77,18493.376218,2.531664,97.402597
7,CHOA CHU KANG,52,20839.827524,2.689176,94.230769
11,JURONG EAST,23,22262.938859,3.486536,100.000000
24,YISHUN,85,24049.247794,3.860268,96.470588
23,WOODLANDS,95,25025.220395,3.847090,95.789474
21,TAMPINES,137,25396.048130,2.744829,96.350365
12,JURONG WEST,81,25609.088735,3.478537,93.827160
18,SEMBAWANG,51,25748.757966,3.836048,96.078431
9,GEYLANG,37,31308.986486,4.666865,89.189189
0,ANG MO KIO,57,31663.557018,4.924164,91.228070


In [27]:
final_summary = {
    "evaluation_month":
        "2026-10",

    "total_cases":
        int(total_count),

    "overall_mae":
        float(overall_mae),

    "overall_median_ape_pct":
        float(overall_median_ape),

    "accepted_cases":
        int(accepted_count),

    "accepted_pct":
        float(accepted_pct),

    "abstained_cases":
        int(abstained_count),

    "abstention_pct":
        float(abstention_pct),

    "accepted_mae":
        float(accepted_mae),

    "accepted_median_ape_pct":
        float(accepted_median_ape),

    "interval_margin":
        float(INTERVAL_MARGIN),

    "interval_width":
        float(interval_width),

    "interval_coverage_pct":
        float(final_interval_coverage)
}

final_summary

{'evaluation_month': '2026-10',
 'total_cases': 2175,
 'overall_mae': 34564.781865517245,
 'overall_median_ape_pct': 3.908201388888889,
 'accepted_cases': 1500,
 'accepted_pct': 68.96551724137932,
 'abstained_cases': 675,
 'abstention_pct': 31.03448275862069,
 'accepted_mae': 32761.142225833337,
 'accepted_median_ape_pct': 3.855751037531577,
 'interval_margin': 65339.25,
 'interval_width': 130678.5,
 'interval_coverage_pct': 88.13333333333333}

In [28]:
OUTPUT_DIR = Path(
    "../models/final"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_DIR
    / "final_evaluation_summary.json",
    "w"
) as file:

    json.dump(
        final_summary,
        file,
        indent=4
    )


final_results.to_csv(
    OUTPUT_DIR
    / "final_evaluation_cases.csv",
    index=False
)

flat_type_results.to_csv(
    OUTPUT_DIR
    / "final_evaluation_by_flat_type.csv",
    index=False
)

town_results.to_csv(
    OUTPUT_DIR
    / "final_evaluation_by_town.csv",
    index=False
)

print(
    "Final evaluation outputs saved."
)

Final evaluation outputs saved.
